# 1. Load Data
Melakukan import libary dan membaca file CSV.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv("train_transaction.csv")
df.head()


,TransactionID,isFraud,TransactionDT,TransactionAmt,ProductCD,card1,card2,card3,card4,card5,...,V330,V331,V332,V333,V334,V335,V336,V337,V338,V339
0,2987000,0,86400,68.5,W,13926,NaN,150.0,discover,142.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2987001,0,86401,29.0,W,2755,404.0,150.0,mastercard,102.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2987002,0,86469,59.0,W,4663,490.0,150.0,visa,166.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2987003,0,86499,50.0,W,18132,567.0,150.0,mastercard,117.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2987004,0,86506,50.0,H,4497,514.0,150.0,mastercard,102.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


# 2. Memahami Struktur Data
Bertujuan untuk memahami struktur dan kualitas dataset, meliputi pengecekan dimensi data, tipe data setiap fitur, serta identifikasi missing value sebagai dasar dalam menentukan proses preprocessing sebelum pemodelan machine learning.
* print(df.shape) → Digunakan untuk melihat jumlah baris dan kolom pada DataFrame.
* print(df.info()) → Digunakan untuk menampilkan informasi umum mengenai dataset.
* print(df.isnull().sum().sort_values(ascending=False).head(20)) → Digunakan untuk mengecek jumlah missing value (nilai kosong) pada setiap kolom.

In [ ]:
print(df.shape)
print(df.info())
print(df.isnull().sum().sort_values(ascending=False).head(20))

(590540, 394)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 590540 entries, 0 to 590539
Columns: 394 entries, TransactionID to V339
dtypes: float64(376), int64(4), object(14)
memory usage: 1.7+ GB
None
dist2    552913
D7       551623
D13      528588
D14      528353
D12      525823
D6       517353
D9       515614
D8       515614
V153     508595
V149     508595
V141     508595
V146     508595
V154     508595
V162     508595
V142     508595
V158     508595
V161     508595
V157     508595
V138     508595
V139     508595
dtype: int64


# 3. Menghitung Persentase Missing Value
Mengidentifikasi kolom yang memiliki persentase data kosong tertinggi sehingga dapat ditentukan langkah preprocessing yang tepat, seperti menghapus kolom, melakukan imputasi, atau mempertahankan kolom tersebut.
* df.isnull().sum() → menghitung jumlah data kosong pada setiap kolom.
* len(df) → jumlah seluruh baris pada dataset dibagi dan dikali 100 (mengubahnya menjadi persentase).
* sort_values(ascending=False) → mengurutkan dari persentase missing terbesar ke terkecil.
* head(20) → menampilkan 20 kolom dengan missing value tertinggi.

In [ ]:
missing_percent = (
    df.isnull().sum() / len(df)
) * 100

missing_percent.sort_values(
    ascending=False
).head(20)

dist2    93.628374
D7       93.409930
D13      89.509263
D14      89.469469
D12      89.041047
D6       87.606767
D9       87.312290
D8       87.312290
V153     86.123717
V149     86.123717
V141     86.123717
V146     86.123717
V154     86.123717
V162     86.123717
V142     86.123717
V158     86.123717
V161     86.123717
V157     86.123717
V138     86.123717
V139     86.123717
dtype: float64

# 4. Menghapus Kolom
Kolom yang memiliki persentase missing value lebih dari 80% dihapus karena sebagian besar datanya kosong, sehingga berpotensi menurunkan kualitas analisis dan kinerja model machine learning.
* df = df.fillna(0) → mengisi semua nilai yang kosong (missing value/NaN) pada dataset dengan angka 0.


In [ ]:
cols_to_drop = missing_percent[
    missing_percent > 80
].index

df = df.drop(
    columns=cols_to_drop
)


In [ ]:
df = df.fillna(0)

# 5. Label Encoding
Mengonversi data kategorikal menjadi data numerik agar dapat digunakan oleh algoritma machine learning yang umumnya hanya menerima input berupa angka.
* df.select_dtypes(include='object').columns → memilih semua kolom yang bertipe object (teks).
* LabelEncoder().fit_transform(df[col].astype(str)) → akan mengubah kategori menjadi angka.

In [ ]:
from sklearn.preprocessing import LabelEncoder
for col in df.select_dtypes(
    include='object'
).columns:

    df[col] = LabelEncoder().fit_transform(
        df[col].astype(str)
    )


# 6. Verifikasi Hasil Preprocessing
Digunakan untuk menampilkan daftar kolom yang masih bertipe object (teks/string).

In [ ]:
print(
    df.select_dtypes(
        include='object'
    ).columns.tolist()
)

print(df.shape)

[]
(590540, 339)


# 7. Persiapan Data untuk Pemodelan
Menentukan variabel target, menganalisis distribusi kelas, serta memisahkan fitur dan target sebagai persiapan sebelum proses pelatihan model machine learning.
* target = "isFraud" → Menentukan bahwa kolom isFraud adalah target yang akan diprediksi.
* print(df["isFraud"].value_counts()) → digunakan untuk melihat jumlah data pada setiap kelas.
* X = df.drop → Membuat variabel X yang berisi seluruh fitur/predictor, kecuali kolom isFraud.
* y = df["isFraud"] → Membuat variabel y yang berisi target yang akan diprediksi.

In [ ]:
target = "isFraud"

In [ ]:
print(
    df["isFraud"].value_counts()
)

isFraud
0    569877
1     20663
Name: count, dtype: int64


In [ ]:
X = df.drop(
    "isFraud",
    axis=1
)

y = df["isFraud"]

# 8. Pembagian Data Training, Testing, dan Standarisasi Data
Membagi dataset menjadi data latih dan data uji serta melakukan standarisasi fitur agar data siap digunakan pada proses pelatihan dan evaluasi model machine learning.
* X_train, X_test, y_train, y_test = train_test_split → 80% data digunakan untuk training (X_train, y_train) dan 20% data digunakan untuk testing (X_test, y_test).
* stratify=y → menjaga proporsi kelas isFraud=0 dan isFraud=1 tetap seimbang pada data train dan test.
* scaler = StandardScaler() → Mengubah skala data sehingga rata-rata (mean) ≈ 0 dan standar deviasi ≈ 1.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train = scaler.fit_transform(
    X_train
)
X_test = scaler.transform(
    X_test
)

# 9. Hyperparameter Tuning Menggunakan Optuna
Mengoptimalkan hyperparameter dan arsitektur Neural Network secara otomatis menggunakan Optuna agar diperoleh model dengan performa terbaik.
Optuna akan mencoba beberapa kombinasi:
* n1 = trial.suggest_int("layer1", 32, 128) → jumlah neuron pada hidden layer pertama (layer1).
* n2 = trial.suggest_int("layer2", 16, 64) → jumlah neuron pada hidden layer kedua (layer2).
* dropout = trial.suggest_float("dropout", 0.1, 0.5) → nilai dropout.
* model = Sequential([...]) → Membangun model Neural Network.
* history = model.fit(...) → Training model.
* return min(history.history['val_loss']) → Optuna akan mencari kombinasi parameter yang menghasilkan validation loss terkecil.


In [ ]:
def objective(trial):

    n1 = trial.suggest_int("layer1", 32, 128)
    n2 = trial.suggest_int("layer2", 16, 64)
    dropout = trial.suggest_float("dropout", 0.1, 0.5)

    model = Sequential([
        Dense(
            n1,
            activation='relu',
            input_shape=(X_train.shape[1],)
        ),

        Dropout(dropout),

        Dense(
            n2,
            activation='relu'
        ),

        Dense(
            1,
            activation='sigmoid'
        )
    ])

    model.compile(
        optimizer='adam',
        loss='binary_crossentropy',
        metrics=['accuracy']
    )

    history = model.fit(
        X_train,
        y_train,
        validation_split=0.2,
        epochs=5,
        batch_size=256,
        verbose=0
    )

    return min(history.history['val_loss'])

* study = optuna.create_study(direction="minimize") → Menjalankan proses tuning.
* study.optimize(objective, n_trials=5) → membuat studi optimasi, mencoba 5 kombinasi hyperparameter, dan memilih kombinasi terbaik.

In [ ]:
import optuna

study = optuna.create_study(direction="minimize")

study.optimize(
    objective,
    n_trials=5
)

[I 2026-06-24 11:14:07,213] A new study created in memory with name: no-name-82c88508-ad91-4eae-86bf-26e42c6e7cdc
c:\Users\Fidela Risyunira\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
[I 2026-06-24 11:14:30,252] Trial 0 finished with value: 0.09791934490203857 and parameters: {'layer1': 111, 'layer2': 60, 'dropout': 0.44689751246289844}. Best is trial 0 with value: 0.09791934490203857.
[I 2026-06-24 11:14:51,608] Trial 1 finished with value: 0.09617689251899719 and parameters: {'layer1': 119, 'layer2': 47, 'dropout': 0.4765449282080817}. Best is trial 1 with value: 0.09617689251899719.
[I 2026-06-24 11:15:10,552] Trial 2 finished with value: 0.09176654368638992 and parameters: {'layer1

# 10. Pelatihan dan Prediksi Model Neural Network
Melatih model Neural Network menggunakan Early Stopping untuk mencegah overfitting, kemudian melakukan prediksi terhadap data uji untuk memperoleh hasil klasifikasi.
* early_stop = EarlyStopping() → menghentikan training jika performa model pada data validasi tidak membaik selama 5 epoch berturut-turut dan mengembalikan bobot (weights) terbaik yang pernah diperoleh selama training.
* X_train, y_train → data training.
* validation_split=0.2 → 20% data training digunakan sebagai validasi.
* epochs=10 → maksimal 10 kali iterasi.
* batch_size=256 → data diproses per 256 sampel.
* callbacks=[early_stop] → menggunakan Early Stopping.
* y_prob = model.predict(X_test) → Menghasilkan probabilitas suatu transaksi termasuk fraud.

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(patience=5,
    restore_best_weights=True
)

history = model.fit(
    X_train,
    y_train,
    validation_split=0.2,
    epochs=10,
    batch_size=256,
    callbacks=[early_stop],
    verbose=1
    )

Epoch 1/10
1477/1477 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.9777 - loss: 0.0822 - val_accuracy: 0.9778 - val_loss: 0.0824
Epoch 2/10
1477/1477 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - accuracy: 0.9780 - loss: 0.0814 - val_accuracy: 0.9780 - val_loss: 0.0819
Epoch 3/10
1477/1477 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - accuracy: 0.9781 - loss: 0.0808 - val_accuracy: 0.9782 - val_loss: 0.0804
Epoch 4/10
1477/1477 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - accuracy: 0.9782 - loss: 0.0805 - val_accuracy: 0.9782 - val_loss: 0.0805
Epoch 5/10
1477/1477 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - accuracy: 0.9782 - loss: 0.0802 - val_accuracy: 0.9781 - val_loss: 0.0806
Epoch 6/10
1477/1477 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - accuracy: 0.9784 - loss: 0.0795 - val_accuracy: 0.9780 - val_loss: 0.0807
Epoch 7/10
1477/1477 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - accuracy: 0.9784 - loss: 0.0791 - val_accuracy: 0.9780 - val_loss: 0.0804
Epoch 8/10
1477/1477 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.9786 - loss: 0.0786 - 

In [ ]:
y_prob =  model.predict(X_test)
y_pred = (y_prob >0.5).astype(int)

3691/3691 ━━━━━━━━━━━━━━━━━━━━ 4s 946us/step


# 11. Evaluasi Kinerja Model
Mengevaluasi performa model klasifikasi menggunakan beberapa metrik, yaitu Accuracy, Precision, Recall, F1-Score, dan ROC AUC guna mengetahui seberapa baik model dalam mendeteksi transaksi fraud.
* accuracy_score(y_test, y_pred) → Mengukur persentase prediksi yang benar dari seluruh data.
* precision_score(y_test, y_pred) → Mengukur seberapa banyak prediksi fraud yang benar-benar fraud.
* recall_score(y_test, y_pred) → Mengukur seberapa banyak kasus fraud yang berhasil dideteksi oleh model.
* f1_score(y_test, y_pred) → Merupakan rata-rata harmonis antara Precision dan Recall.
* roc_auc_score(y_test, y_prob) → Mengukur kemampuan model membedakan kelas fraud dan non-fraud berdasarkan probabilitas prediksi.



In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

accuracy = accuracy_score(
    y_test,
    y_pred
)

precision = precision_score(
    y_test,
    y_pred
)

recall = recall_score(
    y_test,
    y_pred
)

f1 = f1_score(
    y_test,
    y_pred
)

auc = roc_auc_score(
    y_test,
    y_prob
)

print("Accuracy :", accuracy)
print("Precision:", precision)
print("Recall   :", recall)
print("F1 Score :", f1)
print("ROC AUC  :", auc)

Accuracy : 0.9783164561249026
Precision: 0.9110878661087866
Recall   : 0.4214856036777159
F1 Score : 0.5763440860215053
ROC AUC  : 0.9119765016534299


# 12. Experiment Tracking Menggunakan MLflow
Melakukan experiment tracking menggunakan MLflow, yaitu menyimpan dan mendokumentasikan hasil evaluasi model machine learning agar setiap percobaan dapat dipantau dan dibandingkan dengan mudah.

In [ ]:
import mlflow

mlflow.set_tracking_uri(
    "sqlite:///mlflow.db"
)

mlflow.set_experiment(
    "Fraud Detection DL"
)

2026/06/24 11:16:43 INFO mlflow.tracking.fluent: Experiment with name 'Fraud Detection DL' does not exist. Creating a new experiment.


<Experiment: artifact_location=('file:///c:/Users/Fidela Risyunira/Documents/PROJECT/UAS '
 'ML/finalterm-machine-learning/mlruns/2'), creation_time=1782274603800, effective_trace_archival_retention=None, experiment_id='2', last_update_time=1782274603800, lifecycle_stage='active', name='Fraud Detection DL', tags={}, trace_location=None, workspace='default'>

In [ ]:
with mlflow.start_run():

    mlflow.log_metric(
        "Accuracy",
        accuracy
    )

    mlflow.log_metric(
        "Precision",
        precision
    )

    mlflow.log_metric(
        "Recall",
        recall
    )

    mlflow.log_metric(
        "F1",
        f1
    )

    mlflow.log_metric(
        "ROC_AUC",
        auc
    )